# Phase 1 — Day 5: PyTorch Basics
**Date:** 2026-04-23

Welcome to the final day of Phase 1! Today we leave pure data wrangling behind and step into deep learning territory with PyTorch.

### Learning Objectives
- Create and manipulate PyTorch tensors
- Understand autograd and automatic differentiation
- Build a tiny neural network with `nn.Module`
- Define a loss function and optimizer
- Run a complete training loop

In [ ]:
# Setup — install PyTorch if needed, then import
# In Colab or a fresh env, uncomment the pip install line:
# !pip install torch

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Sample data: a simple linear relationship with noise
# y = 3x + 2 + noise
np.random.seed(42)
torch.manual_seed(42)

X_np = np.random.rand(200, 1).astype(np.float32) * 10  # 200 points between 0 and 10
y_np = (3 * X_np + 2 + np.random.randn(200, 1).astype(np.float32) * 1.5)

print(f"X shape: {X_np.shape}, y shape: {y_np.shape}")
print(f"First 5 X values: {X_np[:5].flatten()}")
print(f"First 5 y values: {y_np[:5].flatten()}")

## 1. Tensors — The Building Block

A tensor is PyTorch's version of a NumPy array, but with two superpowers: it can live on a GPU, and it can track gradients for automatic differentiation.

You can create tensors from Python lists, NumPy arrays, or using built-in constructors. They have a `dtype` and a `device` (cpu or cuda), just like NumPy arrays have dtypes.

In [ ]:
# Creating tensors in different ways

# From a Python list
t1 = torch.tensor([1, 2, 3, 4])
print(f"From list:    {t1}, dtype={t1.dtype}")

# From a NumPy array
t2 = torch.from_numpy(X_np[:5])
print(f"From numpy:   {t2.flatten()}, dtype={t2.dtype}")

# Built-in constructors
t3 = torch.zeros(2, 3)
t4 = torch.ones(2, 3)
t5 = torch.randn(2, 3)  # standard normal
t6 = torch.arange(0, 10, 2)

print(f"\nzeros:\n{t3}")
print(f"\nones:\n{t4}")
print(f"\nrandn:\n{t5}")
print(f"\narange: {t6}")

In [ ]:
# Tensor properties and basic operations

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(f"Shape: {a.shape}")
print(f"Dtype: {a.dtype}")
print(f"Device: {a.device}")

# Math operations work element-wise, just like NumPy
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]])
print(f"\na + b = {a + b}")
print(f"a * b = {a * b}")       # element-wise multiply
print(f"a @ b = {a @ b}")       # matrix multiply
print(f"a.mean() = {a.mean()}")
print(f"a.sum(dim=0) = {a.sum(dim=0)}")  # sum along rows (dim=0)

In [ ]:
# Reshaping tensors
c = torch.arange(12)
print(f"Original: {c}, shape={c.shape}")

c_reshaped = c.reshape(3, 4)
print(f"\nReshaped to (3,4):\n{c_reshaped}")

c_viewed = c.view(4, 3)  # view() is like reshape but requires contiguous memory
print(f"\nViewed as (4,3):\n{c_viewed}")

# -1 means "figure it out"
c_auto = c.reshape(2, -1)
print(f"\nReshape with -1 -> (2,6):\n{c_auto}")

## 2. Autograd — Automatic Differentiation

This is where PyTorch gets interesting. When you set `requires_grad=True` on a tensor, PyTorch records every operation you do with it. Then you can call `.backward()` and it computes all the gradients automatically.

Think of it as PyTorch building a computation graph behind the scenes. When you call `.backward()`, it walks backward through that graph using the chain rule to compute derivatives. This is the engine that powers neural network training.

In [ ]:
# Simple autograd example
# Let's compute the derivative of y = x^2 + 3x at x = 5
# dy/dx = 2x + 3 = 2*5 + 3 = 13

x = torch.tensor(5.0, requires_grad=True)
y = x**2 + 3*x

print(f"x = {x}")
print(f"y = x^2 + 3x = {y}")

# Compute gradients
y.backward()

print(f"dy/dx at x=5: {x.grad}")  # should be 13.0

In [ ]:
# Autograd with multiple variables
# z = 2*a^3 + b^2
# dz/da = 6*a^2, dz/db = 2*b

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
z = 2 * a**3 + b**2

z.backward()

print(f"z = 2*a^3 + b^2 = {z.item():.1f}")
print(f"dz/da at a=2: {a.grad.item():.1f} (expected: {6 * 2**2})")
print(f"dz/db at b=3: {b.grad.item():.1f} (expected: {2 * 3})")

In [ ]:
# Detaching from the computation graph
# Sometimes you want to use a tensor's value without tracking gradients

x = torch.tensor(3.0, requires_grad=True)
y = x * 2

# .detach() creates a new tensor that shares data but has no grad history
y_detached = y.detach()
print(f"y requires_grad: {y.requires_grad}")
print(f"y_detached requires_grad: {y_detached.requires_grad}")

# torch.no_grad() context manager is another way
with torch.no_grad():
    y_no_grad = x * 2
    print(f"y_no_grad requires_grad: {y_no_grad.requires_grad}")

## 3. Building a Neural Network with nn.Module

In PyTorch, you define neural networks by subclassing `nn.Module`. You put your layers in `__init__` and define how data flows through them in `forward()`.

PyTorch provides common layers like `nn.Linear` (fully connected), `nn.ReLU`, `nn.Sigmoid`, etc. You stack them together to build your network. The beauty is that PyTorch handles all the gradient computation for you.

In [ ]:
# A simple linear regression model (1 input -> 1 output)
class SimpleLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # 1 input feature, 1 output
    
    def forward(self, x):
        return self.linear(x)

model = SimpleLinearModel()
print(model)
print(f"\nParameters:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.data}, requires_grad={param.requires_grad}")

In [ ]:
# A slightly bigger model: 2-layer network for classification
class TwoLayerNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

net = TwoLayerNet(input_size=3, hidden_size=8, output_size=1)
print(net)
print(f"\nTotal parameters: {sum(p.numel() for p in net.parameters())}")

# Quick test: pass random data through it
dummy_input = torch.randn(5, 3)  # 5 samples, 3 features
dummy_output = net(dummy_input)
print(f"\nInput shape:  {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}")

## 4. Loss Functions and Optimizers

A **loss function** measures how wrong your model's predictions are. An **optimizer** updates the model's parameters to reduce that loss.

Common loss functions: `nn.MSELoss()` for regression, `nn.CrossEntropyLoss()` for classification. Common optimizers: `optim.SGD` (stochastic gradient descent) and `optim.Adam` (usually the default choice).

In [ ]:
# Loss functions in action
predictions = torch.tensor([2.5, 0.0, 2.1])
targets = torch.tensor([3.0, -0.5, 2.0])

# MSE Loss: mean of squared differences
mse_loss = nn.MSELoss()
loss_val = mse_loss(predictions, targets)
print(f"MSE Loss: {loss_val.item():.4f}")

# Manual check: ((2.5-3)^2 + (0-(-0.5))^2 + (2.1-2)^2) / 3
manual = ((2.5-3)**2 + (0-(-0.5))**2 + (2.1-2)**2) / 3
print(f"Manual:   {manual:.4f}")

# Cross Entropy Loss for classification
ce_loss = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 0.5, 0.1],   # model thinks class 0
                        [0.1, 2.5, 0.3]])  # model thinks class 1
labels = torch.tensor([0, 1])  # correct!
print(f"\nCross Entropy Loss: {ce_loss(logits, labels).item():.4f}")

## 5. The Training Loop — Putting It All Together

Here's the core pattern you'll use in every PyTorch project. It has four steps that repeat in a loop:

1. **Forward pass** — feed data through the model to get predictions
2. **Compute loss** — compare predictions to actual targets
3. **Backward pass** — compute gradients with `.backward()`
4. **Update weights** — optimizer adjusts parameters with `.step()`

You also need `optimizer.zero_grad()` before each backward pass, because PyTorch accumulates gradients by default.

In [ ]:
# Complete training loop: linear regression on our sample data
# Remember: y = 3x + 2 (with noise)

# Convert data to tensors
X_train = torch.from_numpy(X_np)
y_train = torch.from_numpy(y_np)

# Create model, loss, optimizer
model = SimpleLinearModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

# Training loop
num_epochs = 200
losses = []

for epoch in range(num_epochs):
    # 1. Forward pass
    y_pred = model(X_train)
    
    # 2. Compute loss
    loss = criterion(y_pred, y_train)
    losses.append(loss.item())
    
    # 3. Backward pass
    optimizer.zero_grad()  # clear old gradients!
    loss.backward()        # compute new gradients
    
    # 4. Update weights
    optimizer.step()
    
    # Print progress every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# Check learned parameters (should be close to weight=3, bias=2)
w = model.linear.weight.item()
b = model.linear.bias.item()
print(f"\nLearned: y = {w:.2f}x + {b:.2f}")
print(f"Actual:  y = 3.00x + 2.00")

In [ ]:
# Let's try Adam optimizer and compare
model2 = SimpleLinearModel()
criterion2 = nn.MSELoss()
optimizer2 = optim.Adam(model2.parameters(), lr=0.1)  # Adam can handle larger learning rates

losses_adam = []
for epoch in range(200):
    y_pred = model2(X_train)
    loss = criterion2(y_pred, y_train)
    losses_adam.append(loss.item())
    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()

w2 = model2.linear.weight.item()
b2 = model2.linear.bias.item()
print(f"Adam result: y = {w2:.2f}x + {b2:.2f}")
print(f"SGD result:  y = {w:.2f}x + {b:.2f}")
print(f"\nAdam final loss:  {losses_adam[-1]:.4f}")
print(f"SGD final loss:   {losses[-1]:.4f}")
print("\nAdam usually converges faster, especially with the right learning rate.")

## Tricky Bits

These are the mistakes everyone makes when starting with PyTorch. Learn them now so you don't debug them at 2am later.

In [ ]:
# MISTAKE 1: Forgetting optimizer.zero_grad()
# Gradients ACCUMULATE by default. If you don't zero them, they keep adding up.

x = torch.tensor(2.0, requires_grad=True)

# First backward
y = x ** 2
y.backward()
print(f"After 1st backward: grad = {x.grad}")  # 4.0

# Second backward WITHOUT zeroing
y = x ** 2
y.backward()
print(f"After 2nd backward (no zero): grad = {x.grad}")  # 8.0! (4 + 4)

# Fix: zero the gradient first
x.grad.zero_()
y = x ** 2
y.backward()
print(f"After zeroing + backward: grad = {x.grad}")  # 4.0 again

In [ ]:
# MISTAKE 2: Mixing up tensor types
# PyTorch is strict about dtypes. Neural networks need float tensors.

try:
    model_test = nn.Linear(2, 1)
    int_input = torch.tensor([1, 2])  # int64 by default!
    output = model_test(int_input)
except RuntimeError as e:
    print(f"Error with int input: {e}")

# Fix: use float
float_input = torch.tensor([1.0, 2.0])  # float32
output = model_test(float_input)
print(f"\nWorks with float: {output}")

In [ ]:
# MISTAKE 3: Wrong tensor shapes
# nn.Linear(in_features, out_features) expects shape (batch_size, in_features)

layer = nn.Linear(3, 1)  # expects 3 features

try:
    bad_input = torch.randn(3)  # shape (3,) -- is this 3 features or batch of 3?
    # This actually works but can cause confusion
    out = layer(bad_input)
    print(f"Shape (3,) -> output shape: {out.shape}")
except Exception as e:
    print(f"Error: {e}")

# Best practice: always include the batch dimension
good_input = torch.randn(5, 3)  # 5 samples, 3 features -- clear!
out = layer(good_input)
print(f"Shape (5,3) -> output shape: {out.shape}  # much clearer!")

## Trick Questions

Test your understanding before moving on to the exercises.

**Q1:** What happens if you call `.backward()` twice on the same computation without `retain_graph=True`?

<details><summary>Answer</summary>
You get a RuntimeError. PyTorch frees the computation graph after the first `.backward()` call to save memory. If you need to call backward twice (rare), pass `retain_graph=True` to the first call.
</details>

**Q2:** If `x = torch.tensor([1, 2, 3])`, what is `x.dtype`?

<details><summary>Answer</summary>
`torch.int64`. Python integers default to int64, not float. This will break if you try to pass it through a neural network. Use `torch.tensor([1.0, 2.0, 3.0])` or `torch.tensor([1, 2, 3], dtype=torch.float32)`.
</details>

**Q3:** What does `torch.no_grad()` do and when should you use it?

<details><summary>Answer</summary>
It temporarily disables gradient tracking. Use it during inference (prediction) or evaluation. It makes your code faster and uses less memory because PyTorch doesn't need to build the computation graph.
</details>

**Q4:** You train for 100 epochs and the loss goes UP instead of down. What's likely wrong?

<details><summary>Answer</summary>
Your learning rate is too high. The optimizer overshoots the minimum on each step. Try reducing it by 10x (e.g., from 0.1 to 0.01). Another possibility: you forgot `optimizer.zero_grad()` and gradients are accumulating.
</details>

**Q5:** What's the difference between `model.parameters()` and `model.named_parameters()`?

<details><summary>Answer</summary>
`model.parameters()` returns just the parameter tensors. `model.named_parameters()` returns tuples of (name, parameter), which is useful for debugging or freezing specific layers.
</details>

## Exercises

Fill in the `___` blanks and run each cell. The `assert` statements will tell you if you got it right.

In [ ]:
# Exercise 1: Create a 3x4 tensor of zeros
result = ___
assert result.shape == (3, 4), f"Expected shape (3, 4), got {result.shape}"
assert result.sum() == 0, "All values should be zero"
print("Exercise 1 passed!")

In [ ]:
# Exercise 2: Convert this NumPy array to a float32 PyTorch tensor
arr = np.array([1, 2, 3, 4, 5])
tensor_result = ___
assert tensor_result.dtype == torch.float32, f"Expected float32, got {tensor_result.dtype}"
assert tensor_result.tolist() == [1.0, 2.0, 3.0, 4.0, 5.0]
print("Exercise 2 passed!")

In [ ]:
# Exercise 3: Compute the gradient of y = 3*x^2 at x=4
# dy/dx = 6x, so at x=4 the answer should be 24.0
x = torch.tensor(4.0, requires_grad=True)
y = ___  # write the formula
y.backward()
assert abs(x.grad.item() - 24.0) < 0.001, f"Expected 24.0, got {x.grad.item()}"
print("Exercise 3 passed!")

In [ ]:
# Exercise 4: Create a neural network with:
#   - Input size: 5
#   - Hidden layer: 10 neurons + ReLU
#   - Output: 1 neuron
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = ___  # Linear layer: 5 -> 10
        self.relu = ___    # ReLU activation
        self.layer2 = ___  # Linear layer: 10 -> 1
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

my_net = MyNet()
test_input = torch.randn(3, 5)
test_output = my_net(test_input)
assert test_output.shape == (3, 1), f"Expected shape (3, 1), got {test_output.shape}"
print("Exercise 4 passed!")

In [ ]:
# Exercise 5: Fix this broken training loop (3 bugs to find)
torch.manual_seed(0)
fix_model = nn.Linear(1, 1)
fix_criterion = nn.MSELoss()
fix_optimizer = optim.SGD(fix_model.parameters(), lr=0.01)

X_fix = torch.randn(50, 1)
y_fix = 2 * X_fix + 1

for epoch in range(100):
    pred = fix_model(X_fix)
    loss = fix_criterion(pred, y_fix)
    # BUG 1: what's missing before backward?
    ___
    loss.backward()
    # BUG 2: what's missing after backward?
    ___

# Check it learned something reasonable
w_fix = fix_model.weight.item()
b_fix = fix_model.bias.item()
assert abs(w_fix - 2.0) < 0.3, f"Weight should be ~2.0, got {w_fix:.2f}"
assert abs(b_fix - 1.0) < 0.3, f"Bias should be ~1.0, got {b_fix:.2f}"
print(f"Exercise 5 passed! Learned: y = {w_fix:.2f}x + {b_fix:.2f}")

In [ ]:
# Exercise 6: Reshape this tensor to shape (2, 6) then compute the mean of each row
t = torch.arange(12, dtype=torch.float32)
reshaped = ___  # reshape to (2, 6)
row_means = ___  # mean of each row (hint: use dim parameter)
assert reshaped.shape == (2, 6), f"Expected (2, 6), got {reshaped.shape}"
assert row_means.shape == (2,), f"Expected shape (2,), got {row_means.shape}"
assert abs(row_means[0].item() - 2.5) < 0.01, f"First row mean should be 2.5"
print("Exercise 6 passed!")

In [ ]:
# Exercise 7: Train a model on y = -x + 5 using Adam optimizer
# Fill in the missing parts
torch.manual_seed(42)
X_ex7 = torch.randn(100, 1)
y_ex7 = -1 * X_ex7 + 5  # target: weight=-1, bias=5

model_ex7 = nn.Linear(1, 1)
criterion_ex7 = ___  # MSE loss
optimizer_ex7 = ___  # Adam with lr=0.1

for epoch in range(300):
    pred = model_ex7(X_ex7)
    loss = criterion_ex7(pred, y_ex7)
    optimizer_ex7.zero_grad()
    loss.backward()
    optimizer_ex7.step()

w7 = model_ex7.weight.item()
b7 = model_ex7.bias.item()
assert abs(w7 - (-1.0)) < 0.1, f"Weight should be ~-1.0, got {w7:.2f}"
assert abs(b7 - 5.0) < 0.1, f"Bias should be ~5.0, got {b7:.2f}"
print(f"Exercise 7 passed! Learned: y = {w7:.2f}x + {b7:.2f}")

### Solutions

<details><summary>Exercise 1</summary>

```python
result = torch.zeros(3, 4)
```
</details>

<details><summary>Exercise 2</summary>

```python
tensor_result = torch.tensor(arr, dtype=torch.float32)
# or: torch.from_numpy(arr).float()
```
</details>

<details><summary>Exercise 3</summary>

```python
y = 3 * x ** 2
```
</details>

<details><summary>Exercise 4</summary>

```python
self.layer1 = nn.Linear(5, 10)
self.relu = nn.ReLU()
self.layer2 = nn.Linear(10, 1)
```
</details>

<details><summary>Exercise 5</summary>

```python
# BUG 1: add optimizer.zero_grad()
fix_optimizer.zero_grad()
# BUG 2: add optimizer.step()
fix_optimizer.step()
```
</details>

<details><summary>Exercise 6</summary>

```python
reshaped = t.reshape(2, 6)
row_means = reshaped.mean(dim=1)
```
</details>

<details><summary>Exercise 7</summary>

```python
criterion_ex7 = nn.MSELoss()
optimizer_ex7 = optim.Adam(model_ex7.parameters(), lr=0.1)
```
</details>

## Cumulative Review Exercises

These cover topics from Days 1-4 (Pandas, NumPy, Data Cleaning, Faker/Python Core). Keep those skills sharp!

In [ ]:
import pandas as pd

# Review 1 (Pandas): Filter rows where 'score' > 80 and select only 'name' and 'score' columns
df_rev = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'score': [95, 72, 88, 65],
    'grade': ['A', 'C', 'B', 'D']
})
result_rev1 = ___  # filter and select
assert len(result_rev1) == 2, f"Expected 2 rows, got {len(result_rev1)}"
assert list(result_rev1.columns) == ['name', 'score']
print("Review 1 passed!")

In [ ]:
# Review 2 (NumPy): Create a 3x3 array and compute the sum along each column (axis=0)
arr_rev2 = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
col_sums = ___  # sum along axis=0
assert col_sums.tolist() == [12, 15, 18], f"Expected [12, 15, 18], got {col_sums.tolist()}"
print("Review 2 passed!")

In [ ]:
# Review 3 (Pandas): Use groupby to find the mean score per department
df_rev3 = pd.DataFrame({
    'name': ['A', 'B', 'C', 'D', 'E'],
    'dept': ['Sales', 'Sales', 'Tech', 'Tech', 'Tech'],
    'score': [80, 90, 70, 85, 95]
})
dept_means = ___  # groupby dept, get mean of score
assert abs(dept_means.loc['Sales'] - 85.0) < 0.01
assert abs(dept_means.loc['Tech'] - 83.33) < 0.01
print("Review 3 passed!")

In [ ]:
# Review 4 (Data Cleaning): Drop duplicate rows and fill NaN with 0
df_rev4 = pd.DataFrame({
    'x': [1, 2, 2, 3, None],
    'y': [10, 20, 20, 30, 40]
})
cleaned = ___  # drop duplicates, then fillna with 0
assert len(cleaned) == 4, f"Expected 4 rows after dedup, got {len(cleaned)}"
assert cleaned['x'].isna().sum() == 0, "Should have no NaN values"
print("Review 4 passed!")

In [ ]:
# Review 5 (NumPy): Broadcasting - multiply a (3,1) array by a (1,4) array
a_rev = np.array([[1], [2], [3]])      # shape (3,1)
b_rev = np.array([[10, 20, 30, 40]])   # shape (1,4)
result_rev5 = ___  # multiply them
assert result_rev5.shape == (3, 4), f"Expected (3,4), got {result_rev5.shape}"
assert result_rev5[2, 3] == 120, f"Expected 120 at [2,3], got {result_rev5[2,3]}"
print("Review 5 passed!")

In [ ]:
# Review 6 (Python Core): Write a list comprehension to get squares of even numbers from 1-10
squares_of_evens = ___  # [4, 16, 36, 64, 100]
assert squares_of_evens == [4, 16, 36, 64, 100]
print("Review 6 passed!")

In [ ]:
# Review 7 (Data Cleaning): Convert a column to datetime
df_rev7 = pd.DataFrame({'date_str': ['2024-01-15', '2024-02-20', '2024-03-25']})
df_rev7['date'] = ___  # convert date_str to datetime
assert df_rev7['date'].dtype == 'datetime64[ns]'
print("Review 7 passed!")

In [ ]:
# Review 8 (Pandas): Use .loc to select rows 1-3 and columns 'name' and 'score'
df_rev8 = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'score': [90, 85, 78, 92, 88],
    'age': [25, 30, 35, 28, 22]
})
subset = ___  # use .loc with index 1:3 and column names
assert subset.shape == (3, 2), f"Expected (3, 2), got {subset.shape}"
assert list(subset.columns) == ['name', 'score']
print("Review 8 passed!")

### Cumulative Review Solutions

<details><summary>Review 1</summary>

```python
result_rev1 = df_rev.loc[df_rev['score'] > 80, ['name', 'score']]
```
</details>

<details><summary>Review 2</summary>

```python
col_sums = arr_rev2.sum(axis=0)
```
</details>

<details><summary>Review 3</summary>

```python
dept_means = df_rev3.groupby('dept')['score'].mean()
```
</details>

<details><summary>Review 4</summary>

```python
cleaned = df_rev4.drop_duplicates().fillna(0)
```
</details>

<details><summary>Review 5</summary>

```python
result_rev5 = a_rev * b_rev
```
</details>

<details><summary>Review 6</summary>

```python
squares_of_evens = [x**2 for x in range(1, 11) if x % 2 == 0]
```
</details>

<details><summary>Review 7</summary>

```python
df_rev7['date'] = pd.to_datetime(df_rev7['date_str'])
```
</details>

<details><summary>Review 8</summary>

```python
subset = df_rev8.loc[1:3, ['name', 'score']]
```
</details>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                  PyTorch Basics Cheat Sheet                  ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  TENSORS                                                     ║
║  torch.tensor([1,2,3])       Create from list                ║
║  torch.from_numpy(arr)       Create from NumPy               ║
║  torch.zeros(m, n)           Zeros matrix                    ║
║  torch.ones(m, n)            Ones matrix                     ║
║  torch.randn(m, n)           Random normal                   ║
║  t.shape, t.dtype, t.device  Properties                      ║
║  t.reshape(m, n)             Reshape (-1 = auto)             ║
║                                                              ║
║  AUTOGRAD                                                    ║
║  requires_grad=True          Enable gradient tracking        ║
║  y.backward()                Compute gradients               ║
║  x.grad                      Access gradient value           ║
║  x.grad.zero_()              Zero out gradients              ║
║  torch.no_grad()             Disable tracking (inference)    ║
║  y.detach()                  Detach from graph               ║
║                                                              ║
║  NEURAL NETWORKS                                             ║
║  class MyModel(nn.Module)    Subclass nn.Module              ║
║  nn.Linear(in, out)          Fully connected layer           ║
║  nn.ReLU()                   Activation function             ║
║  model.parameters()          Get all parameters              ║
║                                                              ║
║  LOSS & OPTIMIZER                                            ║
║  nn.MSELoss()                Regression loss                 ║
║  nn.CrossEntropyLoss()       Classification loss             ║
║  optim.SGD(params, lr)       Stochastic gradient descent     ║
║  optim.Adam(params, lr)      Adam optimizer (usually best)   ║
║                                                              ║
║  TRAINING LOOP                                               ║
║  1. y_pred = model(X)        Forward pass                    ║
║  2. loss = criterion(y_pred, y)  Compute loss                ║
║  3. optimizer.zero_grad()    Clear old gradients             ║
║  4. loss.backward()          Compute new gradients           ║
║  5. optimizer.step()         Update weights                  ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---

**Phase 1 complete!** You now have Python, Pandas, NumPy, data cleaning, and PyTorch basics under your belt.

**Next up: Day 6 — TrainTestAndPipelines** (Phase 2: Classical ML begins! Train/test splits, cross-validation, and sklearn Pipelines.)